# 8.17 — Named entity recognition and sequence labeling

NER is sequence labeling with memory: each token gets a tag, but the tag must also make sense as part of a span.

BIO tags turn entity spans into token labels. Emissions provide local evidence, while CRF transitions and Viterbi decoding enforce legal paths across the sequence.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



TAGS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
TAG_INDEX = {tag: i for i, tag in enumerate(TAGS)}


def legal_transition(prev, cur):
    if cur.startswith("I-"):
        return prev == "B-" + cur[2:] or prev == cur
    return True


def emission_for_token(token):
    scores = np.zeros(len(TAGS))
    scores[TAG_INDEX["O"]] = 0.2
    if token in {"Ada", "Lin", "Ilya", "Mira", "Sam", "Eve", "Nina", "Raj", "Maya", "Dr", "Lee", "Chen", "Babbage", "Altman"}:
        scores[TAG_INDEX["B-PER"]] = 2.5
        scores[TAG_INDEX["I-PER"]] = 1.0
    if token in {"OpenAI", "DeepMind", "Google", "IBM", "Acme", "Beta", "UN", "Mayo", "Clinic", "Labs", "Corp", "LLC"}:
        scores[TAG_INDEX["B-ORG"]] = 2.5
        scores[TAG_INDEX["I-ORG"]] = 1.0
    if token in {"Paris", "Berlin", "London", "New", "York", "Francisco", "San"}:
        scores[TAG_INDEX["B-LOC"]] = 2.5
        scores[TAG_INDEX["I-LOC"]] = 1.0
    return scores


def transition_matrix():
    A = np.zeros((len(TAGS), len(TAGS)))
    for i, prev in enumerate(TAGS):
        for j, cur in enumerate(TAGS):
            A[i, j] = 0.2 if legal_transition(prev, cur) else -5.0
            if prev.startswith("B-") and cur == "I-" + prev[2:]:
                A[i, j] = 1.2
            if prev.startswith("I-") and cur == prev:
                A[i, j] = 1.0
    return A


def viterbi_bio(emissions, A, mask=None):
    emissions = np.array(emissions, dtype=float)
    T = emissions.shape[0]
    mask = np.ones(T, dtype=bool) if mask is None else np.array(mask, dtype=bool)
    dp = emissions[0].copy()
    back = []
    for t in range(1, T):
        scores = dp[:, None] + A + emissions[t][None, :]
        back.append(np.argmax(scores, axis=0))
        dp = np.max(scores, axis=0)
        if not mask[t]:
            dp = dp * 0.0
    idx = int(np.argmax(dp))
    path = [idx]
    for bp in reversed(back):
        idx = int(bp[idx])
        path.append(idx)
    path = list(reversed(path))
    return [TAGS[i] for i in path[:int(mask.sum())]]


def spans(tags):
    out = []
    start = None
    typ = None
    for i, tag in enumerate(tags + ["O"]):
        if tag.startswith("B-") or tag == "O" or (tag.startswith("I-") and typ != tag[2:]):
            if start is not None:
                out.append((start, i, typ))
            start = i if tag.startswith("B-") else None
            typ = tag[2:] if tag.startswith("B-") else None
    return set(out)


def entity_f1(pred, gold):
    pred_spans = spans(pred)
    gold_spans = spans(gold)
    tp = len(pred_spans & gold_spans)
    precision = tp / max(1, len(pred_spans))
    recall = tp / max(1, len(gold_spans))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


## The concept, built once: CRF sequence score

A linear-chain CRF scores $s(x,y)=\sum_t e_t(y_t)+\sum_{t=2}^T A_{y_{t-1},y_t}$ and decodes $\hat y=\arg\max_y s(x,y)$. The lesson sentence has four tokens and BIO tags.

In [ ]:

tokens = ["Ada", "works", "at", "OpenAI"]
gold = ["B-PER", "O", "O", "B-ORG"]
entity_marked = [tag != "O" for tag in gold]
print(tokens, gold, entity_marked)
assert len(tokens) == 4
assert gold == ["B-PER", "O", "O", "B-ORG"]
assert sum(entity_marked) == 2


Viterbi uses transitions to avoid illegal BIO paths. Neutral emissions such as $[0.5,0.5]$ should not be treated as confident evidence; context should resolve them.

In [ ]:

row_probs = softmax([2, 0])
neutral = [0.5, 0.5]
emissions = np.vstack([emission_for_token(tok) for tok in tokens])
pred = viterbi_bio(emissions, transition_matrix())
print(np.round(row_probs, 6))
print(pred)
assert np.allclose(np.round(row_probs, 6), [0.880797, 0.119203])
assert neutral == [0.5, 0.5]
assert pred == gold


## The dataset ladder

D1 is Ada works at OpenAI, then clean entities, adjacent entities and padding, CoNLL-style snippets, and D5 longer multi-entity sentences.

In [ ]:

ladder = make_sequence_ladder("8.17")
for rung in ladder:
    examples = rung.get("examples", [(rung["tokens"], rung["tags"])])
    print(rung["name"], "examples=", len(examples), "sample=", examples[0])


In [ ]:

rows = []
metrics = []
for rung in ladder:
    examples = rung.get("examples", [(rung["tokens"], rung["tags"])])
    scores = []
    for tokens, gold in examples:
        emissions = np.vstack([emission_for_token(tok) for tok in tokens])
        pred = viterbi_bio(emissions, transition_matrix())
        scores.append(entity_f1(pred, gold))
    score = sum(scores) / len(scores)
    metrics.append(score)
    rows.append([rung["name"], len(examples), round(score, 3)])
show_table(rows, ["rung", "examples", "entity F1"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    tokens, gold = rung.get("examples", [(rung["tokens"], rung["tags"])])[0]
    mat = np.vstack([emission_for_token(tok) for tok in tokens])
    ax.imshow(mat, aspect="auto", cmap="viridis")
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("tag")
    ax.set_ylabel("token")
axes[5].plot(range(1, 6), metrics, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("entity-span F1")
axes[5].set_xlabel("rung")
fig.tight_layout()


## Pitfall on D5: independent argmax and padding

Independent token argmax can emit illegal I-tags. Padding must be masked so fake tokens do not create fake span starts or endings.

In [ ]:

d5 = ladder[-1]
tokens, gold = d5["examples"][0]
emissions = np.vstack([emission_for_token(tok) for tok in tokens + ["PAD", "PAD"]])
emissions[0, TAG_INDEX["I-PER"]] = 3.5
independent = [TAGS[i] for i in np.argmax(emissions, axis=1)]
masked = [True] * len(tokens) + [False, False]
fixed = viterbi_bio(emissions, transition_matrix(), mask=masked)
print("independent", independent)
print("fixed", fixed)
print("fixed F1", entity_f1(fixed, gold))
assert independent[0].startswith("I-")
assert len(fixed) == len(tokens)
assert entity_f1(fixed, gold) > entity_f1(independent[:len(tokens)], gold)


## Evaluate it + Practice

- Metric: entity-span F1; no-skill baseline predicts O for every token.
- Sanity check: extracted spans should preserve BIO boundaries.
- Ablation: independent argmax can create illegal I-without-B paths.
- Failure signal: padding changes the number of decoded spans.

Practice: add B-DATE and I-DATE tags to D4.

Practice: compute token accuracy and compare it with span F1 on D5.